In [1]:
import networkx as nx
import os

def load_network_data(file_path):
    try:
        G = nx.read_gml(file_path, label='id')
        return G
    except Exception as e:
        print(f"Error loading network data: {e}")
        return None
    
def baseline_community_detection(G):
    communities = nx.community.louvain_communities(G)
    node_membership = {}
    for community_id, community_nodes in enumerate(communities):
        for node in community_nodes:
            node_membership[node] = community_id + 1

    num_communities = len(communities)
    return num_communities, node_membership

In [2]:
import random
import numpy as np

def initialize_population(G, population_size):
    population = []
    nodes = list(G.nodes())

    node_to_idx = {node: idx for idx, node in enumerate(nodes)}
    idx_to_node = {idx: node for node, idx in node_to_idx.items()}

    num_nodes = len(nodes)

    for _ in range(population_size):
        chromosome = [0] * num_nodes

        for idx in range(num_nodes):
            current_node = idx_to_node[idx]
            neighbors = list(G.neighbors(current_node))

            if neighbors:
                chosen_neighbor = random.choice(neighbors)
                chromosome[idx] = node_to_idx[chosen_neighbor]
            else:
                chromosome[idx] = idx
        population.append(chromosome)
    return population, node_to_idx, idx_to_node

In [3]:
def decode_chromosome(chromosome, idx_to_node):
    temp_graph = nx.Graph()

    for idx in idx_to_node:
        temp_graph.add_node(idx_to_node[idx])

    for idx, neighbor_idx in enumerate(chromosome):
        node = idx_to_node[idx]
        neighbor = idx_to_node[neighbor_idx]
        temp_graph.add_edge(node, neighbor)
    communities = list(nx.connected_components(temp_graph))
    return communities

def calculate_fitness(chromosome, G, idx_to_node):
    communities = decode_chromosome(chromosome, idx_to_node)

    if not communities:
        return -1.0
    nx.karate_club_graph()
    return nx.community.modularity(G, communities)

def uniform_crossover(parent1, parent2, crossover_rate = 0.8):
    if random.random() > crossover_rate:
        return parent1.copy(), parent2.copy()
    
    child1 = []
    child2 = []

    for gene1, gene2 in zip(parent1, parent2):
        if random.random() < 0.5:
            child1.append(gene1)
            child2.append(gene2)
        else:
            child1.append(gene2)
            child2.append(gene1)
    return child1, child2

def mutate(chromosome, G, idx_to_node, mutation_rate = 0.1):
    mutated_chromosome = chromosome.copy()

    for idx in range(len(mutated_chromosome)):
        if random.random() < mutation_rate:
            current_node = idx_to_node[idx]
            neighbors = list(G.neighbors(current_node))

            if neighbors:
                chosen_neighbor = random.choice(neighbors)
                node_to_idx = {node: idx for idx, node in idx_to_node.items()}
                mutated_chromosome[idx] = node_to_idx[chosen_neighbor]

    return mutated_chromosome

In [4]:
def tournament_selection(population, fitness_scores, tournament_size = 3):
    tournament_indices = random.sample(range(len(population)), tournament_size)
    best_idx = max(tournament_indices, key=lambda idx: fitness_scores[idx])
    return population[best_idx]

def evolutionary_algorithms(G, idx_to_node, population_size=100, generations=50, crossover_rate=0.8, mutation_rate=0.05, tournament_size=3):
    population, _, _ = initialize_population(G, population_size)
    best_chromosome = None
    best_fitness = -1.0
    history = []

    for gen in range(generations):
        fitness_scores = [calculate_fitness(chromosome, G, idx_to_node) for chromosome in population]
        current_best_idx = np.argmax(fitness_scores)
        current_best_fitness = fitness_scores[current_best_idx]
        if current_best_fitness > best_fitness:
            best_fitness = current_best_fitness
            best_chromosome = population[current_best_idx].copy()
        history.append(best_fitness)
        if (gen + 1) % 10 == 0:
            print(f"Generation {gen + 1}: Best Fitness = {best_fitness:.4f}")

        next_population = []
        next_population.append(best_chromosome.copy())

        while len(next_population) < population_size:
            parent1 = tournament_selection(population, fitness_scores, tournament_size)
            parent2 = tournament_selection(population, fitness_scores, tournament_size)

            child1, child2 = uniform_crossover(parent1, parent2, crossover_rate)
            mutated_child1 = mutate(child1, G, idx_to_node, mutation_rate)
            mutated_child2 = mutate(child2, G, idx_to_node, mutation_rate)

            next_population.append(mutated_child1)
            if len(next_population) < population_size:
                next_population.append(mutated_child2)
        population = next_population
    
    return best_chromosome, best_fitness, history

In [5]:
def run_evalution(dataset_name, dataset_path):
    print("=" * 50)
    print(f"Evaluating dataset: {dataset_name}")

    G = load_network_data(dataset_path)
    if G is None:
        print("Failed to load the network data. Skipping evaluation.")
        return

    base_num_communities, base_node_membership = baseline_community_detection(G)
    print("\nBaseline Community Detection:")
    print(f"Number of communities detected: {base_num_communities}")
    _, _, idx_to_node = initialize_population(G, population_size=1)
    best_chromosome, best_fitness, _ = evolutionary_algorithms(G, idx_to_node, population_size=60, generations=50)
  
    best_communities = decode_chromosome(best_chromosome, idx_to_node)
    best_communities_membership = {}
    for community_id, community_nodes in enumerate(best_communities):
        for node in community_nodes:
            best_communities_membership[node] = community_id + 1

    print(f"Detected Communities: {len(best_communities)}")
    print("Optimal Modularity:", best_fitness)
    print("=" * 50 + "\n")

In [ ]:
networks = {
    "Dolphins": "real-networks/dolphins/dolphins.gml",
    "Football": "real-networks/football/football.gml",
    "Karate": "real-networks/karate/karate.gml",
    "Krebs": "real-networks/krebs/krebs.gml",
    "Word Adjacencies": "extra-networks/adjnoun/adjnoun.gml",
    "Internet": "extra-networks/as-22july06/as-22july06.gml",
    "Neural Network" : "extra-networks/celegansneural/celegansneural.gml",
    "Les Miserables" : "extra-networks/lesmis/lesmis.gml",
    "Books about US Politics" : "extra-networks/polbooks/polbooks.gml",
    "Power Grid" : "extra-networks/power/power.gml"
}

for dataset_name, dataset_path in networks.items():
    run_evalution(dataset_name, dataset_path)